# DocMind — RAG PDF Assistant

Upload a PDF, index it with **Hugging Face embeddings + FAISS**, and ask questions using **Groq**.

### Run in Google Colab
1. Run the installation cell.
2. Open **Secrets** (key icon in the left sidebar) and add `GROQ_API_KEY`. Enable notebook access.
3. Run the setup and app cells in order.
4. Upload a PDF in the Gradio interface and click **Process PDF**.

> Never paste an API key directly into this notebook or commit it to GitHub.

In [6]:
%pip -q install -U langchain langchain-core langchain-community langchain-text-splitters langchain-groq langchain-huggingface sentence-transformers faiss-cpu pypdf gradio

In [8]:
import os
import time
import gradio as gr

from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load the API key securely from Google Colab Secrets.
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Add GROQ_API_KEY in Colab Secrets and enable notebook access.")

MODEL_NAME = "openai/gpt-oss-20b"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

llm = ChatGroq(model=MODEL_NAME, temperature=0.2, api_key=GROQ_API_KEY)
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("LLM and embedding model initialized.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LLM and embedding model initialized.


## Build the PDF index

The PDF is split into overlapping chunks and indexed in FAISS. The retriever returns the four most relevant chunks for each question.

In [9]:
def build_retriever(pdf_path):
    if not pdf_path:
        raise ValueError("Please upload a PDF first.")

    documents = PyPDFLoader(pdf_path).load()
    if not documents:
        raise ValueError("No readable text was found in this PDF.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100,
        separators=["\\n\\n", "\\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    if not chunks:
        raise ValueError("Could not create text chunks from this PDF.")

    vector_store = FAISS.from_documents(chunks, embeddings)
    retriever = vector_store.as_retriever(search_kwargs={"k": 4})
    return retriever, len(documents), len(chunks)

## RAG functions

Answers are grounded in retrieved PDF content. Source page numbers are displayed below the chat.

In [10]:
QA_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful document assistant. Answer using only the supplied context. "
        "If the context does not contain the answer, say: "
        "'This information is not covered in the uploaded document.'"
    ),
    ("human", "Context:\\n{context}\\n\\nQuestion: {question}")
])

def invoke_with_retry(chain, inputs, retries=3, wait_seconds=30):
    for attempt in range(retries):
        try:
            return chain.invoke(inputs)
        except Exception as exc:
            message = str(exc).lower()
            is_rate_limit = (
                "429" in message
                or "rate_limit" in message
                or "resource_exhausted" in message
            )
            if not is_rate_limit or attempt == retries - 1:
                raise
            time.sleep(wait_seconds)

def process_pdf(pdf_path):
    try:
        retriever, page_count, chunk_count = build_retriever(pdf_path)

        # Generate suggested questions; PDF indexing remains successful if this step fails.
        sample_docs = retriever.invoke("What are the main topics in this document?")
        sample_context = "\\n\\n".join(doc.page_content for doc in sample_docs)[:3500]

        suggestion_prompt = ChatPromptTemplate.from_messages([
            ("system", "Generate exactly 6 short questions about the document. "
                      "Return one question per line, without numbering or bullets."),
            ("human", "{context}")
        ])
        suggestion_chain = suggestion_prompt | llm | StrOutputParser()

        try:
            raw = invoke_with_retry(suggestion_chain, {"context": sample_context})
            suggestions = [
                line.strip(" -•0123456789.\\t")
                for line in raw.splitlines()
                if line.strip()
            ][:6]
        except Exception:
            suggestions = []

        defaults = [
            "What is the document about?",
            "What are its main findings?",
            "Which important concepts are explained?",
            "What evidence is presented?",
            "What conclusions are drawn?",
            "What are the key takeaways?",
        ]
        suggestions = (suggestions + defaults)[:6]

        filename = os.path.basename(pdf_path)
        status = f"Indexed {filename}: {page_count} pages and {chunk_count} chunks."
        return retriever, suggestions, status

    except Exception as exc:
        return None, ["—"] * 6, f"PDF processing failed: {exc}"

def answer_question(question, history, retriever):
    history = list(history or [])
    if not question or not question.strip():
        return history, history, ""

    if retriever is None:
        history.extend([
            {"role": "user", "content": question},
            {"role": "assistant", "content": "Upload and process a PDF before asking questions."},
        ])
        return history, history, ""

    try:
        docs = retriever.invoke(question)
        context = "\\n\\n".join(doc.page_content for doc in docs)
        chain = QA_PROMPT | llm | StrOutputParser()
        answer = invoke_with_retry(
            chain,
            {"context": context, "question": question},
        )

        page_numbers = sorted({
            doc.metadata["page"] + 1
            for doc in docs
            if isinstance(doc.metadata.get("page"), int)
        })
        sources = (
            "Retrieved from page(s): " + ", ".join(map(str, page_numbers))
            if page_numbers else "Page information unavailable."
        )

    except Exception as exc:
        answer = f"Sorry, I couldn't generate an answer: {exc}"
        sources = ""

    history.extend([
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ])
    return history, history, sources

def clear_chat():
    return [], [], ""

## Gradio interface

Run this cell to launch the app. The public share link is temporary and intended for testing or demonstrations.

In [12]:
CSS = """
body, .gradio-container { background: #0a0e1a !important; }
.gradio-container { color: #e8eef8 !important; }
"""

with gr.Blocks(
    title="DocMind — AI Document Assistant"
) as demo:
    retriever_state = gr.State(None)
    history_state = gr.State([])

    gr.Markdown("# 📄 DocMind — AI Document Assistant")
    gr.Markdown("Upload a PDF, ask questions, and view the source pages used.")

    with gr.Row():
        pdf_file = gr.File(label="Upload a PDF", file_types=[".pdf"], type="filepath")
        process_button = gr.Button("⚡ Process PDF", variant="primary")

    status = gr.Textbox(
        label="Status",
        value="Upload a PDF and click Process PDF.",
        interactive=False,
    )

    gr.Markdown("### Suggested questions")
    with gr.Row():
        suggestion_buttons = [gr.Button("—") for _ in range(3)]
    with gr.Row():
        suggestion_buttons += [gr.Button("—") for _ in range(3)]

    chatbot = gr.Chatbot(
    label="Conversation",
    height=420
)

    with gr.Row():
        message = gr.Textbox(
            placeholder="Ask a question about your PDF...",
            show_label=False,
            scale=8,
        )
        send_button = gr.Button("Send", variant="primary")
        clear_button = gr.Button("Clear")

    sources = gr.Textbox(label="Source pages", interactive=False, lines=2)

    def handle_process(path):
        retriever, suggestions, status_text = process_pdf(path)
        return (
            retriever,
            status_text,
            *[gr.update(value=question) for question in suggestions],
        )

    process_button.click(
        handle_process,
        inputs=pdf_file,
        outputs=[retriever_state, status, *suggestion_buttons],
    )

    send_button.click(
        answer_question,
        inputs=[message, history_state, retriever_state],
        outputs=[chatbot, history_state, sources],
    ).then(lambda: "", outputs=message)

    message.submit(
        answer_question,
        inputs=[message, history_state, retriever_state],
        outputs=[chatbot, history_state, sources],
    ).then(lambda: "", outputs=message)

    clear_button.click(
        clear_chat,
        outputs=[chatbot, history_state, sources],
    )

    for button in suggestion_buttons:
        button.click(
            answer_question,
            inputs=[button, history_state, retriever_state],
            outputs=[chatbot, history_state, sources],
        )

demo.launch(
    share=True,
    theme=gr.themes.Base(),
    css=CSS
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://751d23b195eca9eee5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
